# Reproduksi SOTANER — *"What do we really know about State of the art NER?"* (LREC 2022)

Reproduksi eksperimen paper Vajjala & Balasubramaniam (LREC 2022) berbasis **satu notebook**,
dengan fungsi berat di modul `helpers/` (adaptasi minimal dari `../SOTANER Windows/` yang sudah terbukti jalan;
hanya `helpers/conll_to_bio.py` dan `helpers/paper_values.py` yang benar-benar baru).

**Struktur notebook (3 section):**

| Section | Isi | Tabel/Figure paper |
|---|---|---|
| **1. Persiapan & Tabel 2** | cek environment, build BIO 4-kolom dari `conll-2012/v4` lokal, reproduksi Tabel 2 | Tabel 2 |
| **2. Black-box** | evaluasi model off-the-shelf: per tipe entitas, per source, per genre, adversarial | Tabel 3, 4, 5, 6 |
| **3. Training NER** | retrain 10 random split + uji-t; cross-genre (single-genre & leave-one-genre-out) | Tabel 7, Tabel 8, Figure 1 |

Setiap tabel hasil punya kolom **Reported on Paper**, **Obtained** (hasil kita), dan **Delta** (`Obtained − Reported`).
Semua tabel diekspor ke `results/SOTANER_reproduction_tables.xlsx` + CSV di akhir Section 3.

**Cara pakai**
1. `conda activate sotaner` lalu jalankan Jupyter dari folder **`SOTANER Notebook Based/`**.
2. Jalankan sel berurutan. Section 1–2 ringan (~1–2 jam total, dominan inferensi CPU).
3. Section 3 berat: atur `FULL_RUN` di sel konfigurasi (`False` = 3 fold + step kecil untuk cek alur; `True` = skala paper).
4. Spark NLP butuh setup Windows sekali jalan (JDK 17, `hadoop.dll`, jar + model offline) — lihat `README.md`.
   Set `RUN_SPARKNLP=False` untuk melewatinya.

**Deviasi utama dari paper** (detail di `README.md`):
data `conll-2012/v4` LDC lokal (divalidasi ke support 11.257 entitas Tabel 3) · retraining spaCy memakai
`config.cfg` rilisan penulis (CNN tok2vec parser, bukan fine-tune `en_core_web_trf`) · Stanza 1.14 / Spark NLP 5.5.3 (model TF era 2020) ·
random split = `KFold(10, shuffle, seed=42)`.

## Sel konfigurasi global

In [1]:
import os, sys, time, json, glob, platform, importlib, warnings, itertools
from pathlib import Path
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np

# --- paths ---
NB_DIR = Path.cwd()
assert NB_DIR.name == "SOTANER Notebook Based", \
    f"Jalankan notebook dari folder 'SOTANER Notebook Based' (cwd sekarang: {NB_DIR})"
PROJECT_ROOT   = NB_DIR.parent
CONLL2012_V4   = PROJECT_ROOT / "conll-2012" / "v4" / "data"

DATA_DIR = NB_DIR / "data"
BIO_DIR = DATA_DIR / "bio"
SPLITS_DIR = DATA_DIR / "bio-splits"
CROSSGENRE_DIR = BIO_DIR / "crossgenre"
PERTURB_DIR = BIO_DIR / "perturb"
SPARK_FMT_DIR = DATA_DIR / "spark-format"
SPACY_FMT_DIR = DATA_DIR / "spacy-fmt"
STANZA_FMT_DIR = DATA_DIR / "stanza-fmt"
RESULTS_DIR = NB_DIR / "results"
MODELS_DIR = NB_DIR / "models"
TMP_DIR = NB_DIR / "tmp"
for d in (DATA_DIR, RESULTS_DIR, MODELS_DIR, TMP_DIR):
    d.mkdir(exist_ok=True)
for sub in ("A", "B", "C", "D"):
    (RESULTS_DIR / sub).mkdir(exist_ok=True)

sys.path.insert(0, str(NB_DIR))
import helpers
from helpers import paper_values as PV
from helpers.report_utils import comparison_table, export_all, f1_of, summarise_folds

# --- run knobs ------------------------------------------------------------
FULL_RUN = False   # Section 3: False = 3 fold + step kecil (cek alur); True = 10 fold skala paper
RUN_SPARKNLP = True    # jalankan kolom Spark NLP (butuh setup offline, lihat README)
SEED = 42

N_FOLDS = 10    if FULL_RUN else 3
SPACY_MAX_STEPS = 20000 if FULL_RUN else 2000
STANZA_MAX_STEPS = 5000  if FULL_RUN else 2000
SPARK_MAX_EPOCHS = 10 if FULL_RUN else 1     # paper tak menyebut jumlah epoch NerDL
SPACY_GPU = os.environ.get("SOTANER_SPACY_GPU", "0") == "1"

GENRES, NEWS, ALL6 = helpers.GENRES, helpers.NEWS, helpers.ALL6
CROSS = helpers.CROSS_GENRES
ENTITY_TYPES = helpers.ENTITY_TYPES
LIBS = ["spacy", "stanza"] + (["sparknlp"] if RUN_SPARKNLP else [])

ALL_TABLES = {}   # {nama_sheet: DataFrame} -> diekspor di akhir Section 3

print("FULL_RUN =", FULL_RUN, "| RUN_SPARKNLP =", RUN_SPARKNLP,
      "| N_FOLDS =", N_FOLDS, "| SEED =", SEED)
print("conll-2012/v4 :", CONLL2012_V4, "->", "OK" if CONLL2012_V4.exists() else "TIDAK ADA (cek path)")

FULL_RUN = False | RUN_SPARKNLP = True | N_FOLDS = 3 | SEED = 42
conll-2012/v4 : d:\OneDrive\Penelitian\NER Methods Comparison\conll-2012\v4\data -> OK


---
# SECTION 1 — Persiapan Environment & Dataset + Tabel 2

Membangun format data yang dipakai seluruh pipeline: **BIO 4 kolom tab-separated**
(`token \t POS \t constituency \t NER-tag`, satu token per baris, baris kosong antar kalimat)
dari checkout **CoNLL-2012 v4** lokal, lalu mereproduksi **Tabel 2** paper
(sanity check F1 model off-the-shelf pada test set standar).

### 1.2 Cek environment

In [2]:
print("Python :", platform.python_version(), "|", sys.executable)
mods = ["spacy", "stanza", "seqeval", "faker", "sklearn", "scipy", "pandas", "numpy", "matplotlib", "openpyxl"]
if RUN_SPARKNLP:
    mods += ["pyspark", "sparknlp"]
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:12s} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"  {m:12s} MISSING -> {e}")
try:
    import torch
    print("  torch        ", torch.__version__, "| CUDA:", torch.cuda.is_available())
except Exception:
    pass
import spacy
print("  spaCy models :", [n for n in ("en_core_web_trf", "en_core_web_lg") if spacy.util.is_package(n)])
print("\nJika ada MISSING: `conda activate sotaner` lalu `pip install <paket>` "
      "(matplotlib/openpyxl wajib; en_core_web_trf: `python -m spacy download en_core_web_trf`).")

Python : 3.12.9 | d:\Conda\envs\sotaner\python.exe
  spacy        3.8.11
  stanza       1.14.0
  seqeval      ?
  faker        ?
  sklearn      1.8.0
  scipy        1.17.0
  pandas       3.0.0
  numpy        2.3.5
  matplotlib   3.11.1
  openpyxl     3.1.5
  pyspark      3.5.9
  sparknlp     ?
  torch         2.10.0+cu130 | CUDA: True
  spaCy models : ['en_core_web_trf', 'en_core_web_lg']

Jika ada MISSING: `conda activate sotaner` lalu `pip install <paket>` (matplotlib/openpyxl wajib; en_core_web_trf: `python -m spacy download en_core_web_trf`).


### 1.3 Dataset — CoNLL-2012 v4 (OntoNotes 5.0, bagian NER)

Layout yang dibaca `helpers.conll_to_bio.build_all`:

```
conll-2012/v4/data/{train,development,test}/data/english/annotations/<genre>/**/*.v4_gold_conll
```

Kolom `*.v4_gold_conll` (0-indexed): `3`=token, `4`=POS, `5`=parse-bit, `10`=NER bracket
(`(TYPE* … *)` / `(TYPE)`), diubah ke IOB2 lewat *state machine* `flag` (sama seperti skrip upstream).
Genre: `bc bn mz nw pt tc wb`. `pt` (Alkitab) tak punya anotasi NE → dikecualikan dari `all6`/`everything`
(paper §3.1 menyebut **enam** sumber).

### 1.4 Build BIO tree dari `conll-2012/v4`

In [4]:
t0 = time.time()
report = helpers.build_all(str(CONLL2012_V4), str(BIO_DIR))
print(f"\nselesai dalam {time.time() - t0:.0f} dtk -> {BIO_DIR}")

train        bc   sents= 10429  entities=  8654
train        bn   sents=  9723  entities= 17573
train        mz   sents=  6911  entities= 10921
train        nw   sents= 15288  entities= 35771
train        pt   sents= 15263  entities=     0
train        tc   sents= 11162  entities=  2233
train        wb   sents=  6411  entities=  6676
train        news sents= 31922  entities= 64265
train        all6 sents= 59924  entities= 81828

development  bc   sents=  1946  entities=  1459
development  bn   sents=  1172  entities=  2172
development  mz   sents=   642  entities=  1232
development  nw   sents=  2054  entities=  4883
development  pt   sents=  1075  entities=     0
development  tc   sents=  1634  entities=   311
development  wb   sents=  1080  entities=  1009
development  news sents=  3868  entities=  8287
development  all6 sents=  8528  entities= 11066

test         bc   sents=  2037  entities=  1697
test         bn   sents=  1252  entities=  2184
test         mz   sents=   780  entiti

### 1.5 Validasi jumlah kalimat / entitas (harus cocok dengan paper)

In [5]:
from helpers.bio_utils import read_bio_file, count_entities, entity_type_counts

rows = []
for split in ["train", "development", "test"]:
    for name in ["news", "all6", *GENRES]:
        p = BIO_DIR / split / f"onto.{name}.ner"
        if p.exists():
            s, t = read_bio_file(str(p))
            rows.append({"split": split, "subset": name,
                         "sentences": len(s), "entities": count_entities(t)})
counts = pd.DataFrame(rows)
display(counts.pivot_table(index="subset", columns="split",
                           values=["sentences", "entities"], sort=False))

s, t = read_bio_file(str(BIO_DIR / "test" / "onto.all6.ner"))
assert len(s) == 8262 and count_entities(t) == 11257, (len(s), count_entities(t))
print(f"OK  test/onto.all6.ner = {len(s)} kalimat / {count_entities(t)} entitas "
      f"== support micro Tabel 3 paper (11.257)")
print("distribusi tipe entitas (test all6):")
display(pd.Series(dict(sorted(entity_type_counts(t).items(), key=lambda x: -x[1]))))

sentences                     entities                     
split      train development    test    train development     test
subset                                                            
news     31922.0      3868.0  3930.0  64265.0      8287.0   8043.0
all6     59924.0      8528.0  8262.0  81828.0     11066.0  11257.0
bc       10429.0      1946.0  2037.0   8654.0      1459.0   1697.0
bn        9723.0      1172.0  1252.0  17573.0      2172.0   2184.0
mz        6911.0       642.0   780.0  10921.0      1232.0   1163.0
nw       15288.0      2054.0  1898.0  35771.0      4883.0   4696.0
pt       15263.0      1075.0  1217.0      0.0         0.0      0.0
tc       11162.0      1634.0  1366.0   2233.0       311.0    380.0
wb        6411.0      1080.0   929.0   6676.0      1009.0   1137.0

OK  test/onto.all6.ner = 8262 kalimat / 11257 entitas == support micro Tabel 3 paper (11.257)
distribusi tipe entitas (test all6):


GPE            2240
PERSON         1988
ORG            1795
DATE           1602
CARDINAL        935
NORP            841
PERCENT         349
MONEY           314
TIME            212
ORDINAL         195
LOC             179
WORK_OF_ART     166
FAC             135
QUANTITY        105
PRODUCT          76
EVENT            63
LAW              40
LANGUAGE         22
dtype: int64

### 1.6 Tabel 2 — *Performance of OntoNotes NER models in the three NLP libraries*

Kolom tabel keluaran kita:

| Kolom | Arti |
|---|---|
| **Reported on Paper** | kolom *Obtained* pada Tabel 2 paper (F1 yang penulis ukur sendiri dengan seqeval): spaCy 89.09 · Stanza 88.71 · Spark NLP 88.60 |
| **Obtained** | F1 micro entity-level (seqeval) hasil menjalankan model stok pada `data/bio/test/onto.all6.ner` |
| **Delta** | `Obtained − Reported on Paper` |

Sebagai catatan, angka "Reported" versi **situs library** (bukan paper): spaCy 90.0 · Stanza 88.8 · Spark NLP 89.97.

Model: `en_core_web_trf` (spaCy), pipeline `en` OntoNotes (Stanza), `onto_bert_base_cased` + `bert_base_cased` (Spark NLP).
Laporan seqeval lengkap (micro + per-tipe) disimpan untuk dipakai lagi di Section 2.

#### 1.7a spaCy + Stanza — muat model sekali, evaluasi test standar

In [6]:
from helpers.eval_blackbox import load_spacy, load_stanza, evaluate_bio, run_blackbox

NLP = load_spacy()
STZ = load_stanza()

STD_TEST = str(BIO_DIR / "test" / "onto.all6.ner")
(RESULTS_DIR / "A").mkdir(exist_ok=True)

t0 = time.time()
std_report = evaluate_bio(STD_TEST, nlp=NLP, stanza_tagger=STZ)   # {'spacy': {...}, 'stanza': {...}}
print(f"eval spaCy + Stanza pada test standar: {time.time() - t0:.0f} dtk")

with open(RESULTS_DIR / "A" / "onto.all6.txt", "w", encoding="utf-8") as fh:
    for m in ("spacy", "stanza"):
        fh.write(f"Classification report for {m.capitalize()} NER:\n{std_report[m]['text']}\n\n")

obt_t2 = {"spacy": f1_of(std_report["spacy"]), "stanza": f1_of(std_report["stanza"])}
print({k: round(v, 2) for k, v in obt_t2.items()})

spaCy: CPU
spaCy model: en_core_web_trf
Stanza NER pipeline loaded (use_gpu=True)
eval spaCy + Stanza pada test standar: 1125 dtk
{'spacy': np.float64(89.19), 'stanza': np.float64(88.24)}


#### 1.7b Spark NLP — mulai sesi Spark & evaluasi test standar (lewati bila `RUN_SPARKNLP=False`)

In [7]:
SPARK = SN_PIPELINE = sn_std_report = None
if RUN_SPARKNLP:
    from helpers import eval_sparknlp
    from helpers.spark_session import start_spark
    SPARK = start_spark(memory="8g")
    print("Spark", SPARK.version)
    SN_PIPELINE = eval_sparknlp.build_pipeline()          # bert_base_cased + onto_bert_base_cased
    sn_std = eval_sparknlp.evaluate_bio_sparknlp(
        [STD_TEST], spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")
    sn_std_report = sn_std["onto.all6"]
    obt_t2["sparknlp"] = f1_of(sn_std_report)
    print("Spark NLP micro-F1:", round(obt_t2["sparknlp"], 2))

spark env: JAVA_HOME=C:\Program Files\Java\jdk-17 (ok)  HADOOP_HOME=C:\hadoop (hadoop.dll ok)
Spark: offline jar spark-nlp-assembly-5.5.3.jar
Spark 3.5.9
BertEmbeddings.load(file:///C:/Users/hanif/cache_pretrained/bert_base_cased)
NerDLModel.load(file:///C:/Users/hanif/cache_pretrained/onto_bert_base_cased)
onto.all6 micro-F1=88.59  (dropped 0 rows)
Spark NLP micro-F1: 88.59


#### 1.7c Rakit Tabel 2

In [8]:
t2 = pd.DataFrame({
    "Reported on Paper": [PV.TABLE2_OBTAINED[l] for l in LIBS],
    "Obtained": [round(obt_t2[l], 2) for l in LIBS],
    "Delta":[round(obt_t2[l] - PV.TABLE2_OBTAINED[l], 2) for l in LIBS],
    "Reported (situs library)": [PV.TABLE2_WEBSITE[l] for l in LIBS],
}, index=[PV.LIB_LABEL[l] for l in LIBS])
t2.index.name = "Library"
ALL_TABLES["Tabel2_library_check"] = t2
display(t2)

,Reported on Paper,Obtained,Delta,Reported (situs library)
Library,,,,
spaCy,89.09,89.19,0.10,90.00
Stanza,88.71,88.24,-0.47,88.80
Spark NLP,88.60,88.59,-0.01,89.97


---
# SECTION 2 — Black-box Experiments

Evaluasi **model off-the-shelf** (tanpa retrain) pada potongan-potongan test set standar.
Semua metrik = F1 micro entity-level (seqeval). Model spaCy + Stanza dari Section 1 dipakai ulang.

### 2.1 Tabel 3 — F-score per tipe entitas

Paper hanya mencetak **8 tipe** (4 tersering: DATE, GPE, ORG, PERSON + 4 terjarang: LANGUAGE, LAW, EVENT, PRODUCT).
Daftar lengkap **18 tipe** ada di *supplementary* (`Results-Details.xlsx`, sheet `Blackbox-DetailedPerformanceTab`)
dan dipakai sebagai "Reported on Paper" di sini; kolom `in paper Table 3` menandai 8 tipe tsb.
Diambil dari laporan seqeval `onto.all6` yang sudah dihitung di sel 1.7.

In [9]:
def type_f1(rep, et):
    return f1_of(rep, label=et)

obt_t3 = {}
for et in ENTITY_TYPES:
    obt_t3[et] = {"spacy": type_f1(std_report["spacy"], et),
                  "stanza": type_f1(std_report["stanza"], et)}
    if sn_std_report is not None:
        obt_t3[et]["sparknlp"] = type_f1(sn_std_report, et)

t3 = comparison_table("Entity type", ENTITY_TYPES, obt_t3, PV.TABLE3_ALL18, libs=LIBS)
t3.insert(0, "in paper Table 3",
          ["yes" if et in helpers.TABLE3_TYPES else "" for et in ENTITY_TYPES])
ALL_TABLES["Tabel3_per_type"] = t3
display(t3)
print("Catatan: tipe langka (LANGUAGE 22, LAW 40, EVENT 63, PRODUCT 76 gold) statistiknya "
      "berisik — paper pun menandai ini.")

,in paper Table 3,spaCy Reported,spaCy Obtained,spaCy Delta,Stanza Reported,Stanza Obtained,Stanza Delta,Spark NLP Reported,Spark NLP Obtained,Spark NLP Delta
Entity type,,,,,,,,,,
CARDINAL,,82.29,82.95,0.66,85.87,85.24,-0.63,85.54,85.54,-0.00
DATE,yes,85.63,86.87,1.24,86.55,85.49,-1.06,85.54,85.48,-0.06
EVENT,yes,74.42,70.31,-4.11,64.96,58.41,-6.55,53.23,52.03,-1.20
FAC,,74.71,75.64,0.93,73.56,70.23,-3.33,74.42,74.13,-0.29
GPE,yes,95.36,95.64,0.28,95.20,95.46,0.26,95.61,95.59,-0.02
LANGUAGE,yes,74.42,66.67,-7.75,60.61,60.61,-0.00,60.61,60.61,-0.00
LAW,yes,67.50,64.00,-3.50,64.79,58.82,-5.97,64.71,64.71,-0.00
LOC,,75.94,76.22,0.28,75.48,76.42,0.94,79.21,79.21,0.00
MONEY,,89.28,82.80,-6.48,89.03,89.03,0.00,87.07,87.07,-0.00


Catatan: tipe langka (LANGUAGE 22, LAW 40, EVENT 63, PRODUCT 76 gold) statistiknya berisik — paper pun menandai ini.


### 2.2 Tabel 4 — performa per *source* (6 sumber OntoNotes)

In [10]:
SRC = ["bn", "mz", "nw", "bc", "tc", "wb"]
src_files = [str(BIO_DIR / "test" / f"onto.{g}.ner") for g in SRC]

src_res, NLP, STZ = run_blackbox(src_files, str(RESULTS_DIR), "A", nlp=NLP, stanza_tagger=STZ)
sn_src = {}
if RUN_SPARKNLP:
    sn_src = eval_sparknlp.evaluate_bio_sparknlp(
        src_files, spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")

obt_t4 = {}
for g in SRC:
    stem = f"onto.{g}"
    obt_t4[g] = {"spacy": f1_of(src_res[stem]["spacy"]), "stanza": f1_of(src_res[stem]["stanza"])}
    if RUN_SPARKNLP:
        obt_t4[g]["sparknlp"] = f1_of(sn_src[stem])

t4 = comparison_table("Source", SRC, obt_t4, PV.TABLE4_SOURCE, libs=LIBS)
ALL_TABLES["Tabel4_per_source"] = t4
display(t4)

onto.bn spacy=91.95 stanza=91.92
onto.mz spacy=86.69 stanza=85.24
onto.nw spacy=90.98 stanza=90.84
onto.bc spacy=88.77 stanza=85.74
onto.tc spacy=76.78 stanza=75.03
onto.wb spacy=83.71 stanza=81.44
onto.bn micro-F1=90.93  (dropped 0 rows)
onto.mz micro-F1=87.73  (dropped 0 rows)
onto.nw micro-F1=90.94  (dropped 0 rows)
onto.bc micro-F1=87.59  (dropped 0 rows)
onto.tc micro-F1=78.11  (dropped 0 rows)
onto.wb micro-F1=80.11  (dropped 0 rows)


,spaCy Reported,spaCy Obtained,spaCy Delta,Stanza Reported,Stanza Obtained,Stanza Delta,Spark NLP Reported,Spark NLP Obtained,Spark NLP Delta
Source,,,,,,,,,
bn,91.64,91.95,0.31,91.82,91.92,0.10,90.93,90.93,-0.00
mz,88.72,86.69,-2.03,85.97,85.24,-0.73,87.73,87.73,0.00
nw,86.14,90.98,4.84,90.87,90.84,-0.03,90.96,90.94,-0.02
bc,91.55,88.77,-2.78,88.35,85.74,-2.61,87.59,87.59,0.00
tc,71.16,76.78,5.62,76.68,75.03,-1.65,78.38,78.11,-0.27
wb,82.81,83.71,0.90,81.20,81.44,0.24,80.11,80.11,-0.00


### 2.3 Tabel 5 — performa per *genre*

Regrouping paper: **News** = `bn + mz + nw`; `bc`, `tc`, `wb` tetap (genre == source, jadi angkanya sama dengan Tabel 4).

In [11]:
news_file = str(BIO_DIR / "test" / "onto.news.ner")
news_res, NLP, STZ = run_blackbox([news_file], str(RESULTS_DIR), "A", nlp=NLP, stanza_tagger=STZ)
sn_news = {}
if RUN_SPARKNLP:
    sn_news = eval_sparknlp.evaluate_bio_sparknlp(
        [news_file], spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="A")

obt_t5 = {"news": {"spacy": f1_of(news_res["onto.news"]["spacy"]),
                   "stanza": f1_of(news_res["onto.news"]["stanza"])}}
if RUN_SPARKNLP:
    obt_t5["news"]["sparknlp"] = f1_of(sn_news["onto.news"])
for g in ["bc", "tc", "wb"]:
    obt_t5[g] = obt_t4[g]

t5 = comparison_table("Genre", ["news", "bc", "tc", "wb"], obt_t5, PV.TABLE5_GENRE, libs=LIBS)
ALL_TABLES["Tabel5_per_genre"] = t5
display(t5)

onto.news spacy=90.63 stanza=90.31
onto.news micro-F1=90.47  (dropped 0 rows)


,spaCy Reported,spaCy Obtained,spaCy Delta,Stanza Reported,Stanza Obtained,Stanza Delta,Spark NLP Reported,Spark NLP Obtained,Spark NLP Delta
Genre,,,,,,,,,
news,90.79,90.63,-0.16,90.41,90.31,-0.10,90.47,90.47,-0.00
bc,88.72,88.77,0.05,88.35,85.74,-2.61,87.59,87.59,0.00
tc,71.16,76.78,5.62,76.68,75.03,-1.65,78.37,78.11,-0.26
wb,82.81,83.71,0.90,81.20,81.44,0.24,80.11,80.11,-0.00


### 2.4 Tabel 6 — *adversarial test sets*

Enam perturbasi (konteks kalimat tak diubah, hanya token entitas ditulis ulang), ber-*seed*:

| ID | Transformasi |
|---|---|
| P1 | PERSON → kata literal **"Dodo"** (tanpa Faker; uji memorisasi) |
| P2 | PERSON → nama Faker **en_US** |
| P3 | PERSON → nama Faker **en_IN** |
| P4 | PERSON → nama **perempuan** Faker **en_TH** |
| P5 | PERSON → nama **perempuan** Faker **en_IN** |
| P6 | GPE → nama tempat Faker **en_IE** |

Tiga tabel: **All** (micro-F1 keseluruhan), **PER** (F1 kelas PERSON, untuk P1–P5), **GPE** (F1 kelas GPE, untuk P6).

In [12]:
from helpers.perturb import make_all_perturbations

pert = make_all_perturbations(STD_TEST, str(PERTURB_DIR), seed=SEED)
pert_files = [pert[f"perturb{i}"] for i in range(1, 7)]

pr_res, NLP, STZ = run_blackbox(pert_files, str(RESULTS_DIR), "B", nlp=NLP, stanza_tagger=STZ)
sn_pert = {}
if RUN_SPARKNLP:
    sn_pert = eval_sparknlp.evaluate_bio_sparknlp(
        pert_files, spark=SPARK, pipeline=SN_PIPELINE,
        tmp_dir=str(SPARK_FMT_DIR), results_dir=str(RESULTS_DIR), tag="B")


def pf1(stem, lib, label):
    if lib == "sparknlp":
        return f1_of(sn_pert[stem], label=label)
    return f1_of(pr_res[stem][lib], label=label)


def base_f1(lib, label):
    rep = sn_std_report if lib == "sparknlp" else std_report[lib]
    return f1_of(rep, label=label)

rows_all, rows_per = {}, {}
rows_all["None"] = {l: base_f1(l, "micro avg") for l in LIBS}
rows_per["None"] = {l: base_f1(l, "PERSON") for l in LIBS}
for i in range(1, 6):
    stem = f"onto.all.test.perturb{i}"
    rows_all[f"P{i}"] = {l: pf1(stem, l, "micro avg") for l in LIBS}
    rows_per[f"P{i}"] = {l: pf1(stem, l, "PERSON") for l in LIBS}
rows_all["P6"] = {l: pf1("onto.all.test.perturb6", l, "micro avg") for l in LIBS}

rows_gpe = {"None": {l: base_f1(l, "GPE") for l in LIBS},
            "P6":   {l: pf1("onto.all.test.perturb6", l, "GPE") for l in LIBS}}

order_all = ["None", "P1", "P2", "P3", "P4", "P5", "P6"]
t6_all = comparison_table("Setting", order_all, rows_all, PV.TABLE6_ALL, libs=LIBS)
t6_all.insert(0, "perturbation", ["(baseline)"] + [PV.PERTURB_DEFS[f"P{i}"] for i in range(1, 7)])

t6_per = comparison_table("Setting (F1 kelas PERSON)", ["None", "P1", "P2", "P3", "P4", "P5"],
                          rows_per, PV.TABLE6_CLASS, libs=LIBS)
paper_gpe = {"None": PV.TABLE6_NONE_GPE, "P6": PV.TABLE6_CLASS["P6"]}
t6_gpe = comparison_table("Setting (F1 kelas GPE)", ["None", "P6"], rows_gpe, paper_gpe, libs=LIBS)

ALL_TABLES["Tabel6_All"] = t6_all
ALL_TABLES["Tabel6_PER"] = t6_per
ALL_TABLES["Tabel6_GPE"] = t6_gpe
display(t6_all); display(t6_per); display(t6_gpe)
print("Temuan paper yang diharapkan muncul: P1 menjatuhkan F1 PERSON (~93 -> ~83) = model menghafal token; "
      "P4 turun ~10 poin PERSON; P6 menjatuhkan F1 GPE (~95 -> ~65); P2/P3/P5 hanya ~1 poin di All.")

perturb1: PERSON literal:Dodo   locale=-      -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb1.ner)
perturb2: PERSON name           locale=en_US  -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb2.ner)
perturb3: PERSON name           locale=en_IN  -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb3.ner)
perturb4: PERSON name_female    locale=en_TH  -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb4.ner)
perturb5: PERSON name_female    locale=en_IN  -> 3400 tokens rewritten  (d:\OneDrive\Penelitian\NER Methods Comparison\SOTANER Notebook Based\data\bio\perturb\onto.all.test.perturb5.ner)
perturb6: GPE    gpe            locale=en_IE  -> 2868 tokens rewr

KeyboardInterrupt: 

---
# SECTION 3 — Training NER Experiments

**BERAT.** Retrain model dari nol pada berbagai split. Dikendalikan sel konfigurasi:

| | `FULL_RUN=False` (default) | `FULL_RUN=True` (skala paper) |
|---|---|---|
| jumlah fold | 3 | 10 |
| spaCy `max_steps` | 2 000 | 20 000 |
| Stanza `max_steps` | 2 000 | 5 000 |
| Spark NLP `maxEpochs` | 1 | 10 |

`FULL_RUN=False` untuk memastikan seluruh alur jalan; angka Tabel 7 skala penuh **hanya valid** dengan `FULL_RUN=True`
(sesi berjam-jam; di GPU 4 GB, Stanza & spaCy dipaksa CPU secara default — set env `SOTANER_SPACY_GPU=1` untuk coba GPU).
Tiap library punya selnya sendiri supaya bisa dijalankan bertahap.

### 3.1 Buat 10 random split (seeded)

In [ ]:
from helpers.splits import make_kfold_splits

sizes = make_kfold_splits(str(BIO_DIR / "onto.everything.ner"), str(SPLITS_DIR),
                          n_splits=10, seed=SEED)
print(f"\nmemakai {N_FOLDS} dari 10 fold (FULL_RUN={FULL_RUN}). "
      f"proporsi ~ {sizes[0][1]/sum(sizes[0][1:]):.0%}/{sizes[0][2]/sum(sizes[0][1:]):.0%}/"
      f"{sizes[0][3]/sum(sizes[0][1:]):.0%} train/dev/test")

### 3.2 Tabel 7 — retrain 10 random split

#### 3.2a spaCy (CNN tok2vec parser dari `config.cfg`)

In [ ]:
from helpers import train_spacy

spacy_fold_f1 = []
for k in range(1, N_FOLDS + 1):
    tr = train_spacy.prep(str(SPLITS_DIR / f"fold{k}_train.ner"), str(SPACY_FMT_DIR))
    dv = train_spacy.prep(str(SPLITS_DIR / f"fold{k}_dev.ner"),   str(SPACY_FMT_DIR))
    te = train_spacy.prep(str(SPLITS_DIR / f"fold{k}_test.ner"),  str(SPACY_FMT_DIR))
    mb = train_spacy.train(tr, dv, str(MODELS_DIR / "spacy-folds" / f"fold{k}"),
                           max_steps=SPACY_MAX_STEPS, gpu_id=(0 if SPACY_GPU else None))
    f1 = train_spacy.evaluate(mb, te, str(RESULTS_DIR / "C" / f"spacy_fold{k}.json"))
    spacy_fold_f1.append(f1)
    print(f"  spaCy fold{k}: F1 = {f1:.2f}")
print("spaCy folds:", [round(x, 2) for x in spacy_fold_f1])

#### 3.2b Stanza (BiLSTM-CRF via `stanza.models.ner_tagger`)

In [ ]:
from helpers import train_stanza

STANZA_GPU = False   # ubah ke True untuk latih di GPU (muat di 4 GB, ~3x lebih cepat)
stanza_fold_f1 = []
for k in range(1, N_FOLDS + 1):
    trj = str(STANZA_FMT_DIR / f"fold{k}_train.json"); train_stanza.bio_to_json(str(SPLITS_DIR / f"fold{k}_train.ner"), trj)
    dvj = str(STANZA_FMT_DIR / f"fold{k}_dev.json");   train_stanza.bio_to_json(str(SPLITS_DIR / f"fold{k}_dev.ner"),   dvj)
    tej = str(STANZA_FMT_DIR / f"fold{k}_test.json");  train_stanza.bio_to_json(str(SPLITS_DIR / f"fold{k}_test.ner"),  tej)
    model = train_stanza.train(trj, dvj, f"en_fold{k}", str(MODELS_DIR / "stanza-folds"),
                               f"fold{k}.pt", max_steps=STANZA_MAX_STEPS, use_gpu=STANZA_GPU)
    res = train_stanza.predict_and_score(model, tej, f"en_fold{k}", use_gpu=STANZA_GPU)
    stanza_fold_f1.append(res["f1"])
    with open(RESULTS_DIR / "C" / f"stanza_fold{k}.txt", "w", encoding="utf-8") as fh:
        fh.write(res["text"])
    print(f"  Stanza fold{k}: F1 = {res['f1']:.2f}")
print("Stanza folds:", [round(x, 2) for x in stanza_fold_f1])

#### 3.2c Spark NLP (NerDL, BERT features) — hanya bila `RUN_SPARKNLP=True`

In [ ]:
spark_fold_f1 = []
if RUN_SPARKNLP:
    from helpers import train_sparknlp
    for k in range(1, N_FOLDS + 1):
        res = train_sparknlp.run_fold(
            str(SPLITS_DIR / f"fold{k}_train.ner"),
            str(SPLITS_DIR / f"fold{k}_dev.ner"),
            str(SPLITS_DIR / f"fold{k}_test.ner"),
            tmp_dir=str(TMP_DIR), spark=SPARK,
            max_epochs=SPARK_MAX_EPOCHS, seed=0)
        spark_fold_f1.append(res["f1"])
        with open(RESULTS_DIR / "C" / f"spark_fold{k}.txt", "w", encoding="utf-8") as fh:
            fh.write(res["text"])
        print(f"  Spark NLP fold{k}: F1 = {res['f1']:.2f}")
    print("Spark NLP folds:", [round(x, 2) for x in spark_fold_f1])
else:
    print("dilewati (RUN_SPARKNLP=False)")

#### 3.2d Rakit Tabel 7 (avg / s.dev / min / max)

In [ ]:
fold_f1 = {"spacy": spacy_fold_f1, "stanza": stanza_fold_f1}
if RUN_SPARKNLP and spark_fold_f1:
    fold_f1["sparknlp"] = spark_fold_f1

t7 = summarise_folds(fold_f1)
t7["paper avg"]   = [PV.TABLE7_SUMMARY[l]["avg"]  for l in fold_f1]
t7["paper sdev"]  = [PV.TABLE7_SUMMARY[l]["sdev"] for l in fold_f1]
t7["delta avg"]   = (t7["avg"] - t7["paper avg"]).round(2)
ALL_TABLES["Tabel7_random_splits"] = t7
display(t7)
if not FULL_RUN:
    print("CATATAN: FULL_RUN=False -> angka ini hanya cek alur, bukan reproduksi Tabel 7.")

### 3.3 Uji-t berpasangan antar library (paper §5.1)

Paper memakai *two-tailed paired-sample t-test* atas 10 nilai F1 fold (via situs eksternal);
di sini pakai `scipy.stats.ttest_rel`. Klaim paper: **spaCy ≈ Spark NLP** (tidak beda signifikan),
**keduanya > Stanza** (p < 0.01).

In [ ]:
from scipy.stats import ttest_rel

rows = []
for a, b in itertools.combinations(fold_f1, 2):
    va, vb = fold_f1[a], fold_f1[b]
    if len(va) > 1:
        stat, p = ttest_rel(va, vb)
        rows.append({
            "pair": f"{PV.LIB_LABEL[a]} vs {PV.LIB_LABEL[b]}",
            "mean diff": round(np.mean(va) - np.mean(vb), 3),
            "t": round(float(stat), 3),
            "p-value": round(float(p), 4),
            "signif (p<0.01)": "ya" if p < 0.01 else "tidak",
        })
tt = pd.DataFrame(rows)
ALL_TABLES["Tabel7_ttest"] = tt
display(tt)
if N_FOLDS < 10:
    print(f"CATATAN: N_FOLDS={N_FOLDS} (<10) -> uji-t hanya representatif penuh dengan FULL_RUN=True.")

### 3.4 Bangun set cross-genre (Tabel 8 + Figure 1)

In [ ]:
from helpers.crossgenre import make_sets
make_sets(str(BIO_DIR), str(CROSSGENRE_DIR))
TEST_MAP = {g: str(CROSSGENRE_DIR / f"{g}_test.ner") for g in CROSS}

### 3.5–3.6 Helper training cross-genre (dipakai Tabel 8 & Figure 1)

Latih **sekali** per konfigurasi, evaluasi ke **empat** test genre.

In [ ]:
from helpers import train_spacy, train_stanza
if RUN_SPARKNLP:
    from helpers import train_sparknlp


def cg_spacy(name, train_ner, dev_ner):
    mb = train_spacy.train(train_spacy.prep(train_ner, str(SPACY_FMT_DIR)),
                           train_spacy.prep(dev_ner, str(SPACY_FMT_DIR)),
                           str(MODELS_DIR / "cg" / f"spacy_{name}"),
                           max_steps=SPACY_MAX_STEPS, gpu_id=(0 if SPACY_GPU else None))
    out = {}
    for g, p in TEST_MAP.items():
        out[g] = train_spacy.evaluate(mb, train_spacy.prep(p, str(SPACY_FMT_DIR)),
                                      str(RESULTS_DIR / "D" / f"spacy_{name}_{g}.json"))
    return out


def cg_stanza(name, train_ner, dev_ner):
    trj = str(STANZA_FMT_DIR / f"cg_{name}_train.json"); train_stanza.bio_to_json(train_ner, trj)
    dvj = str(STANZA_FMT_DIR / f"cg_{name}_dev.json");   train_stanza.bio_to_json(dev_ner, dvj)
    model = train_stanza.train(trj, dvj, f"en_{name}", str(MODELS_DIR / "cg"),
                               f"stanza_{name}.pt", max_steps=STANZA_MAX_STEPS, use_gpu=False)
    out = {}
    for g, p in TEST_MAP.items():
        tej = str(STANZA_FMT_DIR / f"cg_{name}_test_{g}.json"); train_stanza.bio_to_json(p, tej)
        r = train_stanza.predict_and_score(model, tej, f"en_{name}", use_gpu=False)
        out[g] = r["f1"]
    return out


def cg_spark(name, train_ner, dev_ner):
    if not RUN_SPARKNLP:
        return {}
    model, _, bert = train_sparknlp.fit_nerdl(train_ner, str(TMP_DIR), spark=SPARK,
                                              bert=None, max_epochs=SPARK_MAX_EPOCHS)
    out = {}
    for g, p in TEST_MAP.items():
        out[g] = train_sparknlp.score_nerdl(model, p, str(TMP_DIR), SPARK, bert)["f1"]
    return out


def run_cg(name, train_ner, dev_ner):
    res = {"spacy": cg_spacy(name, train_ner, dev_ner),
           "stanza": cg_stanza(name, train_ner, dev_ner)}
    if RUN_SPARKNLP:
        res["sparknlp"] = cg_spark(name, train_ner, dev_ner)
    return res

### 3.5 Tabel 8 — latih hanya pada genre `news`, uji ke 4 genre

In [ ]:
t8_res = run_cg("t8_news", str(CROSSGENRE_DIR / "news_train.ner"), str(CROSSGENRE_DIR / "news_dev.ner"))
obt_t8 = {g: {l: t8_res[l][g] for l in LIBS} for g in CROSS}
t8 = comparison_table("Test genre", CROSS, obt_t8, PV.TABLE8, libs=LIBS)
ALL_TABLES["Tabel8_single_genre"] = t8
display(t8)
print("Temuan paper: drop besar ke genre tak-terlihat; terburuk spaCy pada tc (~83 -> ~52).")

### 3.6 Figure 1 — *Training on Multiple-genres* (leave-one-genre-out)

Untuk tiap genre `held` ∈ {news, bc, tc, wb}: latih pada **3 genre lain**, dev pada `held`,
uji ke **4** test genre. Paper tidak mencetak angka bar-nya → perbandingan kualitatif;
angka kami disimpan + di-chart. "Reported on Paper" (garis titik hitam) dari `Results-Details.xlsx`.

In [ ]:
HELD_TO_TRAIN = {"news": "wb+tc+bc", "bc": "news+wb+tc", "tc": "news+bc+wb", "wb": "news+tc+bc"}
fig1_obt = {}
for held in CROSS:
    fig1_obt[held] = run_cg(f"f1_not_{held}",
                            str(CROSSGENRE_DIR / f"train_not_{held}.ner"),
                            str(CROSSGENRE_DIR / f"dev_{held}.ner"))

# tabel panjang
recs = []
for held in CROSS:
    for test_g in CROSS:
        for l in LIBS:
            recs.append({"held_out (dev)": held, "trained_on": HELD_TO_TRAIN[held],
                         "test_genre": test_g, "library": PV.LIB_LABEL[l],
                         "Obtained": round(fig1_obt[held][l][test_g], 2),
                         "Reported on Paper": PV.FIGURE1[held][test_g].get(l)})
fig1_df = pd.DataFrame(recs)
fig1_df["Delta"] = (fig1_df["Obtained"] - fig1_df["Reported on Paper"]).round(2)
ALL_TABLES["Figure1_cross_genre"] = fig1_df
display(fig1_df.head(12))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=True)
colors = {"spacy": "#4C72B0", "stanza": "#DD8452", "sparknlp": "#55A868"}
x = np.arange(len(CROSS)); w = 0.8 / len(LIBS)
for ax, held in zip(axes.flat, CROSS):
    for j, l in enumerate(LIBS):
        obt = [fig1_obt[held][l][g] for g in CROSS]
        pap = [PV.FIGURE1[held][g].get(l) for g in CROSS]
        pos = x + (j - (len(LIBS) - 1) / 2) * w
        ax.bar(pos, obt, w, label=PV.LIB_LABEL[l], color=colors[l])
        ax.scatter(pos, pap, color="black", s=18, zorder=3)
    ax.set_title(f"train: {HELD_TO_TRAIN[held]}  /  dev: {held}")
    ax.set_xticks(x); ax.set_xticklabels(CROSS); ax.set_ylim(40, 95)
    ax.grid(axis="y", alpha=.3)
axes.flat[0].legend(loc="lower left", fontsize=9)
fig.suptitle("Figure 1 — Cross-genre: train on 3 genres, dev on the 4th "
             "(bar = Obtained, • = Reported on Paper)", fontsize=12)
fig.supxlabel("test genre"); fig.supylabel("F1")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "Figure1_cross_genre.png", dpi=130)
plt.show()

### 3.7 Ekspor semua tabel ke Excel + CSV

In [ ]:
path = export_all(ALL_TABLES, str(RESULTS_DIR))
print("\nSheet:", list(ALL_TABLES))
if RUN_SPARKNLP and SPARK is not None:
    SPARK.stop(); print("Spark session dihentikan.")

---
## Ringkasan deviasi dari paper (lihat `README.md` untuk detail)

1. **Data** — `conll-2012/v4` `*.v4_gold_conll` LDC lokal, divalidasi: test = 11.257 entitas (= support Tabel 3).
2. **spaCy retraining** (Tabel 7/8/Fig 1) memakai `config.cfg` rilisan penulis = **CNN tok2vec transition-based parser**,
   bukan fine-tune `en_core_web_trf`. Evaluasi off-the-shelf (Tabel 2–6) tetap `en_core_web_trf`.
3. **Stanza** 1.14 + model/pretrain `en` terkini (paper ~1.4 + `combined.pt`).
4. **Spark NLP** 5.5.3 + model TF `onto_bert_base_cased`/`bert_base_cased` (era 2020, sesuai paper), bukan 3.1.2.
   `maxEpochs` NerDL tidak disebut paper → default 1 (cek alur) / 10 (FULL_RUN).
5. **Tabel 2 "Reported on Paper"** = kolom *Obtained* paper; angka situs library hanya sebagai kolom catatan.
6. **Random split** = `KFold(10, shuffle=True, random_state=42)` — tafsiran reproducible dari "10 random splits,
   proporsi = standard split".
7. **`FULL_RUN=False`** (default) memakai 3 fold + step diturunkan; angka Tabel 7 skala penuh hanya dengan `FULL_RUN=True`.
8. Inkonsistensi internal paper: spaCy `bc` Tabel 4 (91.55) vs Tabel 5 (88.72); spaCy `GPE` Tabel 3 (95.36) vs Tabel 6 None (95.61).